# 6.4 K-Nearest Neighbours (KNN)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.04_k-nearest-neighbours.ipynb)

Companion code for Section 6.4 of the book. Run the cells in order from top to bottom.

**Example 6.4.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report
from collections import Counter

# ── KNN from scratch ──────────────────────────────────────────────
class KNNScratch:
    def __init__(self, k=5, metric='euclidean'):
        self.k = k; self.metric = metric

    def fit(self, X, y):
        self.X_train = X; self.y_train = y; return self

    def _distance(self, a, b):
        if self.metric == 'euclidean': return np.sqrt(np.sum((a-b)**2))
        elif self.metric == 'manhattan': return np.sum(np.abs(a-b))

    def predict_one(self, x):
        dists = [self._distance(x, xi) for xi in self.X_train]
        k_idx = np.argsort(dists)[:self.k]
        k_labels = [self.y_train[i] for i in k_idx]
        return Counter(k_labels).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

    def score(self, X, y):
        return (self.predict(X) == y).mean()

# Load and prepare data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# ── Find optimal k ────────────────────────────────────────────────
k_values = range(1, 31)
train_scores, test_scores, cv_scores = [], [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='uniform')
    knn.fit(X_tr, y_train)
    train_scores.append(knn.score(X_tr, y_train))
    test_scores.append(knn.score(X_te, y_test))
    cv = cross_val_score(knn, X_tr, y_train, cv=5, scoring='accuracy')
    cv_scores.append(cv.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f'Optimal k (by 5-fold CV): {best_k}')
print(f'Test accuracy at k={best_k}: {test_scores[best_k-1]:.4f}')

# ── Distance metrics comparison ───────────────────────────────────
print('\nDistance metric comparison (k=5):')
for metric in ['euclidean','manhattan','chebyshev','minkowski']:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_tr, y_train)
    print(f'  {metric:12s}: {knn.score(X_te, y_test):.4f}')

# ── Weighted vs uniform voting ─────────────────────────────────────
print('\nWeighting comparison (k=7):')
for weights in ['uniform','distance']:
    knn = KNeighborsClassifier(n_neighbors=7, weights=weights)
    knn.fit(X_tr, y_train)
    print(f'  {weights:10s}: {knn.score(X_te, y_test):.4f}')

# ── Visualise k selection ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Nearest Neighbours Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(k_values, train_scores, 'b-o', ms=5, lw=2, label='Train', alpha=0.8)
ax.plot(k_values, test_scores,  'r-o', ms=5, lw=2, label='Test',  alpha=0.8)
ax.plot(k_values, cv_scores,    'g-o', ms=5, lw=2, label='CV Mean',alpha=0.8)
ax.axvline(best_k, color='green', ls='--', lw=2, label=f'Best k={best_k}')
ax.set_xlabel('k (Number of Neighbours)'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs k\n(Low k=overfit, High k=underfit)')
ax.legend(); ax.grid(alpha=0.3)

# Decision boundary in 2D
from sklearn.decomposition import PCA
ax = axes[1]
X_2d = PCA(n_components=2).fit_transform(X_tr)
knn_2d = KNeighborsClassifier(n_neighbors=best_k)
knn_2d.fit(X_2d, y_train)
h = 0.2
xx, yy = np.meshgrid(np.arange(X_2d[:,0].min()-1, X_2d[:,0].max()+1, h),
                     np.arange(X_2d[:,1].min()-1, X_2d[:,1].max()+1, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
colors = ['#E53935' if c==0 else '#1976D2' for c in y_train]
ax.scatter(X_2d[:,0], X_2d[:,1], c=colors, s=20, alpha=0.7, edgecolors='white', lw=0.5)
ax.set_title(f'KNN Decision Boundary (k={best_k})')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.tight_layout()
plt.savefig('knn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()